Install Dependencies

In [ ]:
pip install openai PyPDF2

Upload Resume

In [ ]:
from google.colab import files
uploaded = files.upload()
print(uploaded)
file_name = list(uploaded.keys())[0]


Read PDF


In [ ]:
import PyPDF2
def read_resume(file_path):
   text = ""
   with open(file_path, "rb") as file:
       reader = PyPDF2.PdfReader(file)
       for page in reader.pages:
           text += page.extract_text()
   return text
print(read_resume(file_name))


Gemini Setup

In [ ]:
pip install google-genai


Gemini API Key Setup

In [ ]:
from google.colab import userdata

In [ ]:
from google import genai
from google.genai.errors import ServerError
client = genai.Client(api_key=userdata.get('GOOGLE_API_KEY'))



Analyze Function

In [ ]:
from google import genai
from pydantic import BaseModel, Field
from typing import List

# 1. Define a heavily detailed structural schema for deeply granular feedback
class DetailedATSAnalysis(BaseModel):
    ats_score: int = Field(..., description="An overall ATS compliance score from 0 to 100.")
    profile_summary: str = Field(..., description="A deep analysis of the candidate's professional level and domain fit.")

    # What the resume did well
    key_strengths: List[str] = Field(..., description="List of 3-5 major highlights, strong impact phrases, or solid sections.")

    # EXPLICIT GAPS & WHAT IS LACKING
    missing_elements: List[str] = Field(
        ...,
        description="Crucial things this resume is completely LACKING (e.g., missing metrics, lack of GitHub link, missing certifications, gaps in technical stack density)."
    )

    # DIRECT IMPROVEMENTS & REWRITES
    actionable_improvements: List[str] = Field(
        ...,
        description="Clear, specific formatting or structural changes needed to improve ATS parsing."
    )
    suggested_bullet_point_rewrites: List[str] = Field(
        ...,
        description="Provide 2-3 examples of weak/passive sentences found in the resume, followed by an optimized version using strong action verbs and quantitative X-Y-Z metrics."
    )

    # Industry Keyword optimization
    recommended_keywords: List[str] = Field(
        ...,
        description="A list of missing industry-standard skills, frameworks, or tools the candidate should integrate to beat ATS keyword filters."
    )

def analyze_resume(text: str) -> str:
    """
    Performs an advanced, granular ATS analysis with actionable bullet point rewrites
    and keyword optimizations using Gemini 3.5.
    """
    # System context combined with the prompt
    prompt = (
        "You are an elite corporate technical recruiter and expert Applicant Tracking System (ATS) engineer. "
        "Analyze the provided resume text with extreme rigor. Contrast it against modern hiring standards, "
        "checking for formatting parsing safety, keyword density, action verb utilization, and quantitative impact.\n\n"
        f"Resume Content:\n{text}"
    )

    try:
        # Request a structured JSON payload bound tightly to our schema
        response = client.models.generate_content(
            model="gemini-3.1-flash-lite",
            contents=prompt,
            config={
                "response_mime_type": "application/json",
                "response_schema": DetailedATSAnalysis,
            }
        )
        return response.text

    except ServerError as e:
        # Safe fallback in case of high-traffic demand spikes
        if e.status_code == 503:
            print("Gemini 3.5 is experiencing high demand. Processing with fallback model...")
            response = client.models.generate_content(
                model="gemini-3.1-flash-lite",
                contents=prompt,
                config={
                    "response_mime_type": "application/json",
                    "response_schema": DetailedATSAnalysis,
                }
            )
            return response.text
        else:
            raise e



Run

In [ ]:
resume_text = read_resume(file_name)
result = analyze_resume(resume_text)
print(result)
